## full_audit

Writes `Data/audit_full_partNN.zip`

Note: this notebook's own manifest size and hash are marked as self-referential because the notebook is saved after the archive manifest is created.

In [1]:
# full_audit — collects the audit input set into numbered zip parts.
# Output: .../My Master's Thesis/Data/audit_full_partNN.zip
# part01 additionally contains contents_manifest.tsv listing every file in every part.
#
# Included: all generation manifests (screen/test/val/omnicontext), training-side val_loss/
# console/manifest files, soft-prompt checkpoints, all converted adapters with their
# checkpoint paths, sanity_rows.json, all score files, dataset + split jsonls, exemplars,
# human-eval files, all notebooks, plus the three code-only
# pulls: qlora raw-checkpoint header slice, OmniGen2 HF snapshot minus weights, and the
# OmniContext metadata export at the pinned revision.
#
# Excluded: images (*.png/*.jpg/...) and raw multi-GB checkpoint binaries
# (model.safetensors, pytorch_model_fsdp.bin); a 3 MB safetensors header slice of
# qlora checkpoint-2000 is collected instead.

from google.colab import drive
drive.mount("/content/drive")

import os, json, glob, shutil, zipfile

ROOT = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis"
DATA = f"{ROOT}/Data"
STAGE = "/content/audit_stage"
CAP_MB = 250

shutil.rmtree(STAGE, ignore_errors=True)
os.makedirs(STAGE)

EXCLUDE_EXT = {".png", ".jpg", ".jpeg", ".webp", ".gif"}
EXCLUDE_NAMES = {"model.safetensors", "pytorch_model_fsdp.bin", "optimizer.bin",
                 "scheduler.bin", "random_states_0.pkl"}

seen = set()
files = []   # (section, abspath, arcname)

def add(section, path, rel=None):
    p = os.path.realpath(path)
    if p in seen or not os.path.isfile(p):
        return
    b = os.path.basename(p)
    if os.path.splitext(b)[1].lower() in EXCLUDE_EXT or b in EXCLUDE_NAMES:
        return
    if "/audit_full_part" in p or "/audit_pulled" in p or b.startswith("Untitled") or ".ipynb_checkpoints" in p:
        return
    seen.add(p)
    files.append((section, p, rel or os.path.relpath(p, ROOT)))

def sect(name, patterns):
    n0 = len(files)
    empty = []
    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if not hits:
            empty.append(pat)
        for p in hits:
            add(name, p)
    n = len(files) - n0
    print(f"[{name}] +{n} files")
    for pat in empty:
        print(f"    WARN pattern matched nothing: {pat}")
    assert n > 0, f"section {name} collected nothing; check its patterns"

# 1) every generation manifest (screen + test + val + omnicontext)
sect("gen_manifests", [f"{DATA}/gen/**/manifest.json"])

# 2) training-side series and logs (lora + qlora + soft prompt) and sp checkpoints
sect("training_side", [
    f"{DATA}/lora/experiments/*/val_loss.jsonl",
    f"{DATA}/lora/experiments/*/console.log",
    f"{DATA}/lora/experiments/*/train_console.log",
    f"{DATA}/lora/experiments/*/manifest.json",
    f"{DATA}/lora/experiments/*/selection.json",
    f"{DATA}/soft_prompt/experiments/*/val_loss.jsonl",
    f"{DATA}/soft_prompt/experiments/*/console.log",
    f"{DATA}/soft_prompt/experiments/*/manifest.json",
    f"{DATA}/soft_prompt/experiments/*/prompt-*.pt",
])

# 3) converted adapters, checkpoint paths preserved
sect("adapters", [f"{DATA}/lora/experiments/*/checkpoint-*/transformer_lora/pytorch_lora_weights.safetensors"])

# 4) probe row list
sect("sanity_rows", [f"{DATA}/**/sanity_rows.json"])

# 5) all judge/score files
sect("scores", [f"{ROOT}/**/scores/*.jsonl"])
sect("embeddings", [f"{DATA}/embeddings/*.npz"])

# 6) dataset + splits + exemplars
names6 = ["grit-entity-new.jsonl", "train.jsonl", "val.jsonl",
          "test.jsonl", "screen.jsonl", "screen_small.jsonl",
          "screen_exclude_from_training.txt", "fewshot_exemplars.jsonl"]
sect("datasets_splits", [f"{ROOT}/**/{n}" for n in names6])

# 7) human eval
sect("human_eval", [f"{ROOT}/**/{n}" for n in
                    ["key.jsonl", "pairs.jsonl", "responses.jsonl", "prereg.txt"]])

# 8) exploratory drafts first so they are labelled apart from the formal set, then the rest
sect("notebooks_drafts", [f"{ROOT}/**/drafts/*.ipynb"])
sect("notebooks", [f"{ROOT}/**/*.ipynb"])
print(f"    (drafts are collected under notebooks_drafts; the formal inventory is "
      f"order the scripts were made.txt)")

# 9) code-only pulls
P = f"{STAGE}/_pulled"
os.makedirs(P, exist_ok=True)
try:
    src = f"{DATA}/lora/experiments/qlora_r8/checkpoint-2000/model.safetensors"
    with open(src, "rb") as f, open(f"{P}/qlora_ckpt2000_head.bin", "wb") as g:
        g.write(f.read(3_000_000))
    add("pulled", f"{P}/qlora_ckpt2000_head.bin", "_pulled/qlora_ckpt2000_head.bin")
    print("[pulled] qlora header slice ok")
except Exception as e:
    print("WARN header slice:", e)

os.system("pip -q install -U huggingface_hub datasets > /dev/null 2>&1")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("OmniGen2/OmniGen2", local_dir=f"{P}/og2_meta",
                      ignore_patterns=["*.safetensors", "*.bin", "*.pt", "*.pth",
                                       "*.onnx", "*.gguf", "*.h5", "*.msgpack"])
    for q in glob.glob(f"{P}/og2_meta/**/*", recursive=True):
        if os.path.isfile(q):
            add("pulled", q, "_pulled/og2_meta/" + os.path.relpath(q, f"{P}/og2_meta"))
    print("[pulled] og2_meta ok")
except Exception as e:
    print("WARN og2_meta:", e)

try:
    import datasets
    REV = "0f6aabbe8d3448d51184093372e980205d9980d3"
    try:
        d = datasets.load_dataset("OmniGen2/OmniContext", split="train", revision=REV)
    except Exception:
        from huggingface_hub import snapshot_download
        pth = snapshot_download("OmniGen2/OmniContext", repo_type="dataset", revision=REV,
                                local_dir="/content/oc_raw", ignore_patterns=["assets/*"])
        try:
            d = datasets.load_from_disk(pth)
        except Exception:
            d = datasets.concatenate_datasets(
                [datasets.Dataset.from_file(x) for x in sorted(glob.glob(pth + "/data-*.arrow"))])
    try:
        d = d.cast_column("input_images", datasets.Sequence(datasets.Image(decode=False)))
    except Exception:
        pass
    with open(f"{P}/oc_all.jsonl", "w", encoding="utf-8") as f:
        for r in d:
            f.write(json.dumps({"key": r["key"], "task_type": r["task_type"],
                                "instruction": r["instruction"],
                                "instruction_language": r.get("instruction_language"),
                                "n_refs": len(r["input_images"])}) + "\n")
    add("pulled", f"{P}/oc_all.jsonl", "_pulled/oc_all.jsonl")
    print("[pulled] oc_all.jsonl rows:", sum(1 for _ in open(f"{P}/oc_all.jsonl")))
except Exception as e:
    print("WARN omnicontext:", e)

# ---- run context, recorded rather than inferred ----
ctx = {
    "omnigen2_code_commit": "18e6f9d5271b517fcb32e999f10df943ae9b8f20",
    "omnicontext_dataset_revision": "0f6aabbe8d3448d51184093372e980205d9980d3",
    "omnigen2_paper_version": "arXiv:2506.18871v3 (27 Sep 2025)",
    "hf_model_revision": "not pinned at run time for the completed runs; recorded as unknown "
                         "rather than backfilled, and pinned for any future run",
    "generation_settings": {"steps": 50, "text_guidance_scale": 5.0, "image_guidance_scale": 2.0,
                            "seed": 0, "max_sequence_length": 1024,
                            "note": "uniform across all 68 generation manifests; size 512 and "
                                    "align_res False are recorded on 66 of them, the two "
                                    "OmniContext baseline manifests carrying their geometry "
                                    "under a different sampler schema"},
    "excluded_from_package": "source and generated image trees; pixel-derived metrics and "
                             "image-to-uid correspondence cannot be replayed from this archive",
    "notebook_inventory": "order the scripts were made.txt",
}
with open(f"{STAGE}/run_context.json", "w") as f:
    json.dump(ctx, f, indent=2)
add("meta", f"{STAGE}/run_context.json", "run_context.json")

# ---- contents manifest + multi-part packing ----
import hashlib

def _sha256(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()

files.sort(key=lambda t: (t[0], t[2]))
man = f"{STAGE}/contents_manifest.tsv"
with open(man, "w") as f:
    f.write("section\trelpath\tbytes\tsha256\n")
    for s, p, r in files:
        # this notebook cannot measure itself: Colab writes this run's output after the
        # manifest is built, so any size or hash recorded here is stale by construction
        if os.path.basename(p) == "full_audit.ipynb":
            f.write(f"{s}\t{r}\tself-referential\tself-referential\n")
        else:
            f.write(f"{s}\t{r}\t{os.path.getsize(p)}\t{_sha256(p)}\n")
files.insert(0, ("meta", man, "contents_manifest.tsv"))

# old parts are removed only here, after every collection step and the manifest have succeeded,
# so an aborted run leaves the previous archive intact
for old in glob.glob(f"{DATA}/audit_full_part*.zip"):
    os.remove(old)

CAP = CAP_MB * 1_000_000
parts = []
zf = None
cur = 0
idx = 0
for s, p, r in files:
    sz = os.path.getsize(p)
    if zf is None or (cur > 0 and cur + sz > CAP):
        if zf:
            zf.close()
        idx += 1
        path = f"{DATA}/audit_full_part{idx:02d}.zip"
        parts.append(path)
        zf = zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED)
        cur = 0
    zf.write(p, arcname=r)
    cur += sz
if zf:
    zf.close()

print("\nWROTE PARTS:")
for pth in parts:
    print("  ", os.path.basename(pth), round(os.path.getsize(pth) / 1e6, 1), "MB")
print("total files:", len(files))


Mounted at /content/drive
[gen_manifests] +68 files
[training_side] +50 files
    WARN pattern matched nothing: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/*/console.log
    WARN pattern matched nothing: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/*/manifest.json
[adapters] +25 files
[sanity_rows] +1 files
[scores] +91 files
[embeddings] +13 files
[datasets_splits] +8 files
[human_eval] +4 files
[notebooks_drafts] +3 files
[notebooks] +150 files
    (drafts are collected under notebooks_drafts; the formal inventory is order the scripts were made.txt)
[pulled] qlora header slice ok


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 34 files:   0%|          | 0/34 [00:00<?, ?it/s]

[pulled] og2_meta ok


README.md:   0%|          | 0.00/4.16k [00:00<?, ?B/s]

...

data-00001-of-00002.arrow: reconstructing file:   0%|          |  0.00B /  420MB            

data-00001-of-00002.arrow: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Casting the dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

[pulled] oc_all.jsonl rows: 400

WROTE PARTS:
   audit_full_part01.zip 228.1 MB
   audit_full_part02.zip 225.1 MB
   audit_full_part03.zip 16.6 MB
   audit_full_part04.zip 54.2 MB
   audit_full_part05.zip 81.4 MB
   audit_full_part06.zip 136.5 MB
total files: 444


In [2]:
# audit_pulled — qlora checkpoint-2000 safetensors header slice, OmniGen2 HF snapshot minus
# weights, OmniContext metadata at revision 0f6aabb. Output: Data/audit_pulled.zip.

from google.colab import drive
drive.mount("/content/drive")

import os, json, glob, shutil

DATA = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
P = "/content/pulled"
shutil.rmtree(P, ignore_errors=True)
os.makedirs(P)

with open(f"{DATA}/lora/experiments/qlora_r8/checkpoint-2000/model.safetensors", "rb") as f, \
     open(f"{P}/qlora_ckpt2000_head.bin", "wb") as g:
    g.write(f.read(3_000_000))
print("header slice written")

os.system("pip -q install -U huggingface_hub datasets > /dev/null 2>&1")
from huggingface_hub import snapshot_download
snapshot_download("OmniGen2/OmniGen2", local_dir=f"{P}/og2_meta",
                  ignore_patterns=["*.safetensors", "*.bin", "*.pt", "*.pth",
                                   "*.onnx", "*.gguf", "*.h5", "*.msgpack"])
print("og2_meta written")

import datasets
REV = "0f6aabbe8d3448d51184093372e980205d9980d3"
try:
    d = datasets.load_dataset("OmniGen2/OmniContext", split="train", revision=REV)
except Exception:
    pth = snapshot_download("OmniGen2/OmniContext", repo_type="dataset", revision=REV,
                            local_dir="/content/oc_raw", ignore_patterns=["assets/*"])
    try:
        d = datasets.load_from_disk(pth)
    except Exception:
        d = datasets.concatenate_datasets(
            [datasets.Dataset.from_file(x) for x in sorted(glob.glob(pth + "/data-*.arrow"))])
try:
    d = d.cast_column("input_images", datasets.Sequence(datasets.Image(decode=False)))
except Exception:
    pass
with open(f"{P}/oc_all.jsonl", "w", encoding="utf-8") as f:
    for r in d:
        f.write(json.dumps({"key": r["key"], "task_type": r["task_type"],
                            "instruction": r["instruction"],
                            "instruction_language": r.get("instruction_language"),
                            "n_refs": len(r["input_images"])}) + "\n")
print("oc_all.jsonl rows:", sum(1 for _ in open(f"{P}/oc_all.jsonl")))

shutil.make_archive(f"{DATA}/audit_pulled", "zip", P)
print("WROTE", f"{DATA}/audit_pulled.zip",
      round(os.path.getsize(f"{DATA}/audit_pulled.zip") / 1e6, 1), "MB")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
header slice written


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 34 files:   0%|          | 0/34 [00:00<?, ?it/s]

og2_meta written
oc_all.jsonl rows: 400
WROTE /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/audit_pulled.zip 32.5 MB


In [3]:
import zipfile, hashlib, glob, csv, collections

DATA = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
parts = sorted(glob.glob(f"{DATA}/audit_full_part*.zip"))
assert parts, "no archive parts found"

where, dup = {}, []
for p in parts:
    with zipfile.ZipFile(p) as z:
        for n in z.namelist():
            if n.endswith("/"): continue
            if n in where: dup.append(n)
            where[n] = p
print(f"parts {len(parts)} | entries {len(where)} | duplicated across parts {len(dup)}")

with zipfile.ZipFile(where["contents_manifest.tsv"]) as z:
    rows = list(csv.reader(z.read("contents_manifest.tsv").decode().splitlines(), delimiter="\t"))
hdr, rows = rows[0], rows[1:]
print(f"manifest rows {len(rows)} | columns {hdr}")

def sha_stream(zf, name, buf=1 << 20):
    h, n = hashlib.sha256(), 0
    with zf.open(name) as fh:
        for chunk in iter(lambda: fh.read(buf), b""):
            h.update(chunk); n += len(chunk)
    return h.hexdigest(), n

missing, size_bad, hash_bad, skipped = [], [], [], []
handles = {p: zipfile.ZipFile(p) for p in parts}
for sec, rel, size, sha in rows:
    if rel not in where: missing.append(rel); continue
    if size == "self-referential": skipped.append(rel); continue
    got_sha, got_size = sha_stream(handles[where[rel]], rel)
    if got_size != int(size): size_bad.append((rel, size, got_size))
    if got_sha != sha: hash_bad.append(rel)
for z in handles.values(): z.close()

extra = sorted(set(where) - {r[1] for r in rows} - {"contents_manifest.tsv"})
print(f"\nmissing from parts       {len(missing)}")
print(f"size mismatches          {len(size_bad)}")
print(f"sha256 mismatches        {len(hash_bad)}")
print(f"self-referential skipped {len(skipped)} {skipped}")
print(f"in parts, not in manifest {len(extra)} {extra[:5]}")
print("\nby section:")
for s, n in sorted(collections.Counter(r[0] for r in rows).items()):
    print(f"  {s:20s} {n}")
ok = not (missing or size_bad or hash_bad or extra or dup)
print("\nARCHIVE VERIFIED" if ok else "\nARCHIVE VERIFICATION FAILED")
assert ok

parts 6 | entries 444 | duplicated across parts 0
manifest rows 443 | columns ['section', 'relpath', 'bytes', 'sha256']

missing from parts       0
size mismatches          0
sha256 mismatches        0
self-referential skipped 1 ['full_audit.ipynb']
in parts, not in manifest 0 []

by section:
  adapters             25
  datasets_splits      8
  embeddings           13
  gen_manifests        68
  human_eval           4
  meta                 1
  notebooks            150
  notebooks_drafts     3
  pulled               29
  sanity_rows          1
  scores               91
  training_side        50

ARCHIVE VERIFIED
